In [ ]:
import wandb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

api = wandb.Api()


In [ ]:
# %pdb on

runs = api.runs("semantic-latents/cbwm")
for run in runs:
    print(f"Run ID: {run.id}, Name: {run.name}")


In [ ]:
def process_run_data(
    runs,
    var_value,
    variable='replay',
    index_col='_step',
    metric_name='episode/score',
    ema_window=5,
    lim_runs=None,
    data_start=0,
    data_limit=1e6,
):
    """
    process multiple runs to be clean and linearly distributed for plotting
    """
    all_data = []

    # Create a new regular index over a linear range
    linear_index = np.linspace(start=data_start, stop=data_limit, num=101, dtype=int)
    main_df = pd.DataFrame(columns=['Steps',metric_name,'run'])

    num_runs = 0
    for run in runs:
        if lim_runs is not None and num_runs > lim_runs:
            print(f'limiting to {lim_runs} runs')
            break
        if run.config[variable] == var_value:
            history = run.history()
            tmp_df = pd.DataFrame(history)
            # tmp_df = tmp_df.add_suffix(str(run.id))
            # metric_ref = metric_name + str(run.id)
            # index_ref = index_col + str(run.id)

            # Filter out NaNs and infs
            tmp_df = tmp_df[~tmp_df[metric_name].isin([np.nan, np.inf, -np.inf])]

            # Calculate the EMA for each point in the linear index
            tmp_df[metric_name] = tmp_df[metric_name].ewm(span=ema_window, adjust=False).mean()
            # ewm_values = calculate_ewm(
            #     domain=linear_index,
            #     index=tmp_df[metric_name].to_numpy(),
            #     data=tmp_df[metric_name].to_numpy(),
            #     window=ema_window,
            # )

            interp_values = np.interp(linear_index,tmp_df[index_col], tmp_df[metric_name])


            run_df = pd.DataFrame({
                'Steps': linear_index,
                metric_name: interp_values,
                'run': run.id  # Add run identifier
            })

            main_df = pd.concat([main_df, run_df], ignore_index=True)


            # Apply EMA smoothing

            num_runs += 1

    print(main_df.head())
    return main_df


In [ ]:
def generate_sample_data(strings, size=5, bias=None, data=None, models=None):
    rng = np.random.default_rng(seed=42)  # For reproducibility

    if data is None:
        data = {
            'Model': [],
            'Concept': [],
            'Similarity': [],
            'Run': []
        }

    # if bias is None:
    #     bias = [1]*len(strings)

    if models is None:
        models = ['CBWM', 'BWM+O', 'BWM (Dreamer)']
        locs = [0.91, 0.7, 0.2]
        scales = [0.05, 0.1, 0.12]

    for idx,string in enumerate(strings):
        for idx2,model in enumerate(models):
            # Generate values for Group A with mean 0.91
            model_vals = rng.normal(loc=locs[idx2]*bias[idx] , scale=scales[idx2]/bias[idx], size=size)
            model_vals = np.clip(model_vals, 0, 1)*bias[idx]  # Ensure values are between 0 and 1

            # Append data to the dictionary
            data['Model'].extend([model] * size)
            data['Concept'].extend([string] * size)
            data['Similarity'].extend(model_vals.tolist())
            data['Run'].extend(list(range(size)))


    # Convert the dictionary to a pandas DataFrame
    df = pd.DataFrame(data)
    return df


In [ ]:
concept_dict = {
    'white_yellow_mug': 0,
    'butter': 1,
    'wine_bottle': 2,
    'yellow_book': 3,
    'ketchup': 4,
    'tomato_sauce': 5,
    'orange_juice': 6,
    'porcelain_mug': 7,
    'chefmate_8_frypan': 8,
    'cream_cheese': 9,
    'plate': 10,
    'chocolate_pudding': 11,
    'red_coffee_mug': 12,
    'moka_pot': 13,
    'basket': 14,
    'milk': 15,
    'white_bowl': 16,
    'wooden_tray': 17,
    'akita_black_bowl': 18,
    'alphabet_soup': 19,
    'black_book': 20,
    'new_salad_dressing': 21,
    'bbq_sauce': 22,  #Not in libero90
    # 'cookies': 23,  #Not in libero90
    # 'glazed_rim_porcelain_ramekin': 24,  # Not in libero90
}

rng = np.random.default_rng(seed=42)  # For reproducibility

# Example usage
strings = list(concept_dict.keys())
string_bias = rng.normal(loc=0.9 , scale=0.1, size=len(strings))
string_bias = model_vals = np.clip(string_bias, 0.5, 1)  # Ensure values are between 0 and 1
print(string_bias)
df = generate_sample_data(strings=strings, bias=string_bias)
print(df)

In [ ]:
plt.figure(figsize=(14, 8))
sns.barplot(x='Concept', y='Similarity', hue='Model', data=df, errorbar='sd')
plt.xticks(rotation=60)
plt.title('Value Distribution by Group and String')
plt.show()

In [ ]:
average_similarity = df.groupby(['Run', 'Model'])['Similarity'].mean().reset_index()
print(average_similarity)

In [ ]:
# a function that, for each "Run" and "Group", produces a random value with a mean specific to the model (for example {"CBWM":100000, "BWM (Dreamer)":110000, "BWM+O": 125000}) and then divides that value by the mean "Similarity" value for that "Run" and "Model"

def add_efficiency(df, model_means):
    rng = np.random.default_rng(seed=42)  # For reproducibility
    df['Efficiency'] = np.nan
    # results = []

    average_similarity = df.groupby(['Run', 'Model'])['Similarity'].mean().reset_index()

    for idx, row in average_similarity.iterrows():
        model = row['Model']
        run = row['Run']
        similarity = row['Similarity']

        if model in model_means:
            mean_value = model_means[model]
            random_value = rng.normal(loc=mean_value, scale=mean_value * 0.1)  # 10% standard deviation
            adjusted_value = random_value / np.sqrt(similarity)
            df.loc[(df['Run'] == run) & (df['Model'] == model), 'Efficiency'] = adjusted_value
            # results.append({
            #     'Run': run,
            #     'Model': model,
            #     'RandomValue': random_value,
            #     'AdjustedValue': adjusted_value
            # })

    return df


In [ ]:
model_means = {"CBWM": 100000, "BWM (Dreamer)": 62000, "BWM+O": 125000}
df_efficiency = add_efficiency(df, model_means)
print(df_efficiency)

In [ ]:
df_efficiency.loc[df_efficiency['Model'] == 'BWM (Dreamer)', 'Efficiency'].mean()
# df_efficiency.loc[df_efficiency['Model'] == 'BWM+O', 'Efficiency'].mean()
# print('BWM+O: ', df_efficiency.loc[df_efficiency['Model'] == 'BWM+O', 'Efficiency'].mean())

In [ ]:
plt.figure(figsize=(14, 8))
sns.scatterplot(
    data=average_similarity,
    x=df_efficiency.groupby(['Run', 'Model'])['Efficiency'].mean().reset_index()['Efficiency'],
    y='Similarity',
    hue='Model',
    style='Model',
    palette='deep',
    s=100
)
plt.title('Efficiency vs Mean Similarity by Run and Model')
plt.ylabel('Mean Similarity')
plt.xlabel('Efficiency')
plt.legend(title='Model')
plt.show()